# Forma Fechada para o Problema X22 — demonstração computacional

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x22_formas_fechadas.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*  
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Objetivo

Ilustrar numericamente a vantagem da forma fechada na solução do Problema X22B10T0.

A solução em série decompõe $\Theta(x,t)$ em três termos:

$$\Theta(x,t) = \frac{\alpha q''_0}{k}\frac{t}{L}
+ \underbrace{\frac{2Lq''_0}{\pi^2 k}\sum_{m=1}^{M}\frac{\cos(m\pi x/L)}{m^2}}_{\text{série estacionária (convergência lenta)}}
- \underbrace{\frac{2Lq''_0}{\pi^2 k}\sum_{m=1}^{M}\frac{\cos(m\pi x/L)}{m^2}\,e^{-A_m t}}_{\text{série transiente (convergência rápida)}}$$

Substituindo a série estacionária pela forma fechada exata:

$$\frac{2L}{\pi^2}\sum_{m=1}^{\infty}\frac{\cos(m\pi x/L)}{m^2} = -x + \frac{x^2}{2L} + \frac{L}{3}$$

obtém-se a solução com forma fechada, que converge com pouquíssimos termos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Parâmetros físicos e autovalores

Os autovalores do Problema X22 têm fórmula explícita: $\beta_m = m\pi$, raízes de $\sin(\beta_m) = 0$.  
O coeficiente $A_m = \alpha\beta_m^2/L^2 = \alpha m^2\pi^2/L^2$ controla a taxa de decaimento exponencial.

In [ ]:
L     = 66e-4        # espessura da placa [m]
alpha = 1.93e-7      # difusividade térmica [m²/s]
k     = 0.81         # condutividade térmica [W/(m·K)]
q0    = 1e3          # fluxo de calor em x=0 [W/m²]

M_max = 500
m_arr = np.arange(1, M_max + 1)
beta  = np.pi * m_arr          # β_m = mπ
A     = alpha * beta**2 / L**2 # A_m = α β_m² / L²

## Painel esquerdo: convergência da série estacionária

O somatório $(2L/\pi^2)\sum_{m=1}^{M}\cos(m\pi x/L)/m^2$ converge lentamente ($\sim 1/m^2$).  
A forma fechada $-x + x^2/(2L) + L/3$ é o valor exato, atingido apenas no limite $M\to\infty$.

In [ ]:
x_norm = np.linspace(0, 1, 400)
x_vals = x_norm * L

cores    = ['#D55E00', '#0072B2', '#009E73', '#CC79A7']
tracados = ['--', '-.', ':', (0, (5, 1))]
M_vals   = [1, 2, 5, 20]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for M_val, cor, ls in zip(M_vals, cores, tracados):
    serie = (2*L/np.pi**2) * np.sum(
        np.cos(np.outer(beta[:M_val], x_norm)) / m_arr[:M_val, None]**2, axis=0
    )
    ax1.plot(x_norm, serie, color=cor, linestyle=ls, linewidth=1.4, label=f'M = {M_val}')

forma_fechada = -x_vals + x_vals**2/(2*L) + L/3
ax1.plot(x_norm, forma_fechada, 'k-', linewidth=1.5, label='Forma fechada')
ax1.set_xlabel('x/L')
ax1.set_ylabel('(2L/π²) Σ cos(mπx/L)/m²')
ax1.set_title('Convergência da série estacionária')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.5)

## Painel direito: T(x=0, t) — série vs. forma fechada

Compara $T(0,t)$ calculada com a série lenta (muitos termos necessários) contra a solução com forma fechada (converge com $M=1$).

In [ ]:
t = np.linspace(0, 500, 600)
x = 0.0

def T_serie(M_val, t_vec):
    """Solução com dois somatórios (sem forma fechada)."""
    mm = m_arr[:M_val]
    Am = A[:M_val]
    ss = (2*L/np.pi**2) * (q0/k) * np.sum(1.0 / mm**2)   # cos(mπ·0)=1
    tr = (2*L/np.pi**2) * (q0/k) * np.sum(
        np.exp(-np.outer(Am, t_vec)) / mm[:, None]**2, axis=0
    )
    return alpha * q0/k * t_vec / L + ss - tr

def T_fechada(M_val, t_vec):
    """Solução com forma fechada + somatório transiente."""
    mm   = m_arr[:M_val]
    Am   = A[:M_val]
    perm = (q0/k) * (-x + x**2/(2*L) + L/3)
    tr   = (2*L/np.pi**2) * (q0/k) * np.sum(
        np.exp(-np.outer(Am, t_vec)) / mm[:, None]**2, axis=0
    )
    return alpha * q0/k * t_vec / L + perm - tr

ax2.plot(t, T_fechada(200, t), 'k-', linewidth=1.5, label='Referência (M=200)')
for M_val, cor, ls in zip([1, 2, 5], cores[:3], tracados[:3]):
    ax2.plot(t, T_serie(M_val, t), color=cor, linestyle=ls, linewidth=1.4,
             label=f'Série, M={M_val}')
ax2.plot(t, T_fechada(1, t), color=cores[3], linestyle=tracados[3],
         linewidth=1.4, label='Forma fechada, M=1')

ax2.set_xlabel('Tempo (s)')
ax2.set_ylabel('T(0, t) (°C)')
ax2.set_title('T(x=0, t): convergência em número de termos')
ax2.legend(loc='upper left')
ax2.grid(True, linestyle='--', alpha=0.5)

fig.tight_layout()
plt.show()